## To prepare a consume the resulting images from the blue print action on Unreal
* Two mask and one image generated, 
    * OG, the original image, that will be used to train the YOLO model
    * BLUE, and X-ray like image identifying where the object is
    * RED, a second X-ray, but with occlusion from the objects on from of the target shoe
    
* Why two x-rays, the blue tells me the total are in the picture that the shoe occupies, if no other object was present, the red, shoes the area that is actually visible, from this i can calculate a visibility score and ignore all images where this score is to low. 
* The red x-ray will allow me to calculate the box for supervised training


<table>
  <tr>
    <td><img src="Images/image_0CE3ADA94EC4E001DF1744A11387B4D0_BLUE.png" alt="BLUE" width="100%"></td>
    <td><img src="Images/image_0CE3ADA94EC4E001DF1744A11387B4D0_OG.png" alt="OG" width="100%"></td>
    <td><img src="Images/image_0CE3ADA94EC4E001DF1744A11387B4D0_RED.png" alt="RED" width="100%"></td>
  </tr>
</table>

In [1]:
"""
prepare_dataset.py

Processes a folder of masked images into a YOLO-ready dataset.

Expected input structure:
    input_dir/
        <name>_OG.png    original image
        <name>_RED.png   mask: visible portion of object (red pixels)
        <name>_BLUE.png  mask: full object extent (blue pixels)

Output structure:
    output_dir/
        images/   original images
        labels/   YOLO .txt labels (empty or with bounding box)

Visibility = red_area / blue_area
    < 20%   → copy image + empty label  (object not present)
    20-60%  → skip                      (ambiguous, not used)
    > 60%   → copy image + YOLO label   (bounding box from RED mask)
"""

'\nprepare_dataset.py\n\nProcesses a folder of masked images into a YOLO-ready dataset.\n\nExpected input structure:\n    input_dir/\n        <name>_OG.png    original image\n        <name>_RED.png   mask: visible portion of object (red pixels)\n        <name>_BLUE.png  mask: full object extent (blue pixels)\n\nOutput structure:\n    output_dir/\n        images/   original images\n        labels/   YOLO .txt labels (empty or with bounding box)\n\nVisibility = red_area / blue_area\n    < 20%   → copy image + empty label  (object not present)\n    20-60%  → skip                      (ambiguous, not used)\n    > 60%   → copy image + YOLO label   (bounding box from RED mask)\n'

In [2]:
import os
import shutil
import numpy as np
from pathlib import Path
from PIL import Image


# ── Configuration ─────────────────────────────────────────────────────────────

INPUT_DIR  = Path("input")       # folder with OG / RED / BLUE images
OUTPUT_DIR = Path("output")      # destination

VISIBILITY_LOW  = 0.20           # below → empty label
VISIBILITY_HIGH = 0.60           # above → yolo label  (between → skip) 

YOLO_CLASS = 0

In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def red_pixels(img_array: np.ndarray) -> np.ndarray:
    """Boolean mask of pixels that are 'red' in an RGB image."""
    r, g, b = img_array[..., 0], img_array[..., 1], img_array[..., 2]
    return (r > 127) & (g < 80) & (b < 80)


def blue_pixels(img_array: np.ndarray) -> np.ndarray:
    """Boolean mask of pixels that are 'blue' in an RGB image."""
    r, g, b = img_array[..., 0], img_array[..., 1], img_array[..., 2]
    return (b > 127) & (r < 80) & (g < 80)


def bounding_box_yolo(mask: np.ndarray, img_h: int, img_w: int) -> tuple:
    """
    Returns YOLO-format bounding box (x_center, y_center, width, height)
    normalised to [0, 1] from a boolean pixel mask.
    """
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    y_min, y_max = np.where(rows)[0][[0, -1]]
    x_min, x_max = np.where(cols)[0][[0, -1]]

    x_center = ((x_min + x_max) / 2) / img_w
    y_center = ((y_min + y_max) / 2) / img_h
    width    = (x_max - x_min) / img_w
    height   = (y_max - y_min) / img_h

    return x_center, y_center, width, height


def ensure_dirs(output: Path):
    (output / "images").mkdir(parents=True, exist_ok=True)
    (output / "labels").mkdir(parents=True, exist_ok=True)

In [6]:
# ── Main ──────────────────────────────────────────────────────────────────────

def process(input_dir: Path, output_dir: Path):
    ensure_dirs(output_dir)

    og_files = sorted(input_dir.glob("*_OG.png"))

    if not og_files:
        print(f"No *_OG.png files found in {input_dir}")
        return

    stats = {"copied_empty": 0, "copied_label": 0, "skipped": 0, "errors": 0}

    for og_path in og_files:
        stem = og_path.stem[: -len("_OG")]          # strip trailing _OG
        red_path  = input_dir / f"{stem}_RED.png"
        blue_path = input_dir / f"{stem}_BLUE.png"

        # ── Validate all three files exist ────────────────────────────────────
        missing = [p for p in (red_path, blue_path) if not p.exists()]
        if missing:
            print(f"[WARN] {stem}: missing {[p.name for p in missing]}, skipping.")
            stats["errors"] += 1
            continue

        # ── Load masks ────────────────────────────────────────────────────────
        try:
            og_img   = Image.open(og_path).convert("RGB")
            red_img  = Image.open(red_path).convert("RGB")
            blue_img = Image.open(blue_path).convert("RGB")
        except Exception as e:
            print(f"[ERROR] {stem}: {e}")
            stats["errors"] += 1
            continue

        img_h, img_w = og_img.size[1], og_img.size[0]

        red_arr  = np.array(red_img)
        blue_arr = np.array(blue_img)

        red_mask  = red_pixels(red_arr)
        blue_mask = blue_pixels(blue_arr)

        red_area  = int(red_mask.sum())
        blue_area = int(blue_mask.sum())

        # ── Visibility ────────────────────────────────────────────────────────
        if blue_area == 0:
            # No object in blue mask at all → treat as not present
            visibility = 0.0
        else:
            visibility = red_area / blue_area

      

        # ── Routing logic ─────────────────────────────────────────────────────
        if visibility < VISIBILITY_LOW:
            # Object not meaningfully visible → copy with empty label
            dest_image = output_dir / "images" / f"empty_{og_path.name}"
            dest_label = output_dir / "labels" / f"empty_{stem}_OG.txt"
        
            shutil.copy2(og_path, dest_image)
            dest_label.write_text("")
            stats["copied_empty"] += 1
            print(f"[EMPTY ] {stem}  vis={visibility:.2%}  red={red_area}px  blue={blue_area}px")

        elif visibility > VISIBILITY_HIGH:
            # Object clearly visible → copy with YOLO bounding box from RED mask
            if red_mask.sum() == 0:
                print(f"[WARN] {stem}: visibility={visibility:.2%} but red mask is empty, skipping.")
                stats["errors"] += 1
                continue

            dest_image = output_dir / "images" / og_path.name
            dest_label = output_dir / "labels" / f"{stem}_OG.txt"

            x_c, y_c, w, h = bounding_box_yolo(red_mask, img_h, img_w)
            shutil.copy2(og_path, dest_image)
            dest_label.write_text(f"{YOLO_CLASS} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")
            stats["copied_label"] += 1
            print(f"[LABEL ] {stem}  vis={visibility:.2%}  bbox=({x_c:.3f},{y_c:.3f},{w:.3f},{h:.3f})")

        else:
            # Ambiguous visibility → skip
            stats["skipped"] += 1
            print(f"[SKIP  ] {stem}  vis={visibility:.2%}  (between {VISIBILITY_LOW:.0%}–{VISIBILITY_HIGH:.0%})")

    # ── Summary ───────────────────────────────────────────────────────────────
    total     = sum(stats.values())
    processed = total - stats["errors"]

    def bar(count, width=40) -> str:
        filled = round((count / total) * width) if total else 0
        return "█" * filled + "░" * (width - filled)

    def pct(count) -> str:
        return f"{count / total * 100:5.1f}%" if total else "  0.0%"

    print(f"""
╔══════════════════════════════════════════════════════╗
║           Dataset Distribution — {total:>4} triplets        ║
╠══════════════════════════════════════════════════════╣
║  Class 0  (vis > {VISIBILITY_HIGH:.0%})  {bar(stats['copied_label'])}  {pct(stats['copied_label'])}  ({stats['copied_label']})
║  Empty    (vis < {VISIBILITY_LOW:.0%})  {bar(stats['copied_empty'])}  {pct(stats['copied_empty'])}  ({stats['copied_empty']})
║  Ignored  ({VISIBILITY_LOW:.0%}–{VISIBILITY_HIGH:.0%})    {bar(stats['skipped'])}  {pct(stats['skipped'])}  ({stats['skipped']})
║  Errors                   {bar(stats['errors'])}  {pct(stats['errors'])}  ({stats['errors']})
╠══════════════════════════════════════════════════════╣
║  Copied to output: {stats['copied_label'] + stats['copied_empty']:>4}  │  Discarded: {stats['skipped']:>4}  │  Errors: {stats['errors']:>3}  ║
╚══════════════════════════════════════════════════════╝""")


In [7]:
INPUT_DIR = Path("C:/Temp/img")
OUTPUT_DIR = Path("./Datasets/Original/Cafe_1")

process(INPUT_DIR, OUTPUT_DIR)

[LABEL ] image_001296A3478C2F90ED768287208F237B  vis=99.81%  bbox=(0.494,0.500,0.037,0.094)
[LABEL ] image_00B460474CD08100DF15F6A70C00A558  vis=60.85%  bbox=(0.501,0.496,0.036,0.075)
[EMPTY ] image_00E51FF64781074DC8E9E99C9705ED4B  vis=0.00%  red=0px  blue=1470px
[SKIP  ] image_018DE47047E3FC2B749812B28FD8C74B  vis=24.05%  (between 20%–60%)
[LABEL ] image_02E53F9048226359523361A2C098282E  vis=99.77%  bbox=(0.498,0.499,0.034,0.144)
[SKIP  ] image_0365EDBF464CC1483192519477DEEE0E  vis=46.84%  (between 20%–60%)
[SKIP  ] image_03B445AA44D202E084F0DDAD54734F59  vis=20.75%  (between 20%–60%)
[EMPTY ] image_03D29373439FA2010DD19196FF9F6C7B  vis=12.63%  red=118px  blue=934px
[LABEL ] image_05FB7F6B4CE51628A2164FA9D0B4E545  vis=83.53%  bbox=(0.499,0.504,0.073,0.068)
[LABEL ] image_0747C4964E75719F6C4FC78E942A893C  vis=99.04%  bbox=(0.495,0.496,0.056,0.130)
[LABEL ] image_083D70F64E43A27B1AEA89B243F92AA1  vis=84.62%  bbox=(0.501,0.497,0.025,0.023)
[LABEL ] image_0892C2A94F5D8E5A10B89AB8DB21D077